In [23]:
import pandas as pd
import numpy as np
import torch
import plotly.graph_objects as go

from src.utils import generate_mask_tensor
from src.embedding import embed
from src.gp_ccm import GP_ccm_sig
from src.iaaft import surrogates

from src.sp_ccm import run_SP_CCM, SP_CCM_iaaft

In [24]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

Using device: cuda



# Generate data

In [25]:
# Set length of timeseries
N_length = torch.tensor([400])

# Initialise values at t = 0
d = torch.tensor([0.2])
e = torch.tensor([0.4])

# Autoregressive function
for t in range(N_length - 1):
    
    d_next = d[t] * (3.78 - (3.78 * d[t]))
    e_next = e[t] * (3.77 - (3.77 * e[t]) - (0.8 * d[t]))

    d = torch.concat((d, d_next.unsqueeze(0)))
    e = torch.concat((e, e_next.unsqueeze(0)))
            
# Normalising step
d_norm = d.sub(d.mean(dim = -1).unsqueeze(-1)).div(d.std(dim = -1).unsqueeze(-1))
e_norm = e.sub(e.mean(dim = -1).unsqueeze(-1)).div(e.std(dim = -1).unsqueeze(-1))

# Try time offset in synchrony

# Set length of timeseries
N_length = torch.tensor([400])

# Initialise values at t = 0
d = torch.tensor([0.2, 0.3])
e = torch.tensor([0.4, 0.3])

# Autoregressive function
for t in range(1, N_length - 1):
    
    d_next = d[t] * (3.78 - (3.78 * d[t-1]))
    e_next = e[t] * (3.77 - (3.77 * e[t]) - (0.8 * d[t-1]))

    d = torch.concat((d, d_next.unsqueeze(0)))
    e = torch.concat((e, e_next.unsqueeze(0)))
            
# Normalising step
d_norm = d.sub(d.mean(dim = -1).unsqueeze(-1)).div(d.std(dim = -1).unsqueeze(-1))
e_norm = e.sub(e.mean(dim = -1).unsqueeze(-1)).div(e.std(dim = -1).unsqueeze(-1))

In [26]:
# Set length of timeseries
N_length = torch.tensor([400])

# Initialise values at t = 0
x = torch.tensor([0.2])
y = torch.tensor([0.4])

# Autoregressive function
for t in range(N_length - 1):
    
    # from ECCM
    x_next = x[t] * (3.8 - (3.8 * x[t]))
    y_next = y[t] * (3.1 - (3.1 * y[t]) - (0.8 * x[t]))

    x = torch.concat((x, x_next.unsqueeze(0)))
    y = torch.concat((y, y_next.unsqueeze(0)))
            
# Normalising step
x_norm = x.sub(x.mean(dim = -1).unsqueeze(-1)).div(x.std(dim = -1).unsqueeze(-1))
y_norm = y.sub(y.mean(dim = -1).unsqueeze(-1)).div(y.std(dim = -1).unsqueeze(-1))

In [27]:
fig = go.Figure()

fig.add_trace(go.Scatter(x = torch.arange(0, x_norm.shape[0]), y = d_norm,
                    mode = 'lines+markers',
                    name = 'D'))

fig.add_trace(go.Scatter(x = torch.arange(0, x_norm.shape[0]), y = e_norm,
                    mode = 'lines+markers',
                    name = 'E'))

fig.update_layout(title = 'Confounding time series"',
                   xaxis_title = 'time',
                   yaxis_title = 'value')

fig.show()

In [28]:
d_norm_iaaft = torch.tensor(surrogates(x = d_norm, ns = 1), dtype = torch.float32).squeeze()
e_norm_iaaft = torch.tensor(surrogates(x = e_norm, ns = 1), dtype = torch.float32).squeeze()

d_norm_iaaft_100 = torch.tensor(surrogates(x = d_norm, ns = 100, tol_pc = 5.), dtype = torch.float32).squeeze()
e_norm_iaaft_100 = torch.tensor(surrogates(x = e_norm, ns = 100, tol_pc = 5.), dtype = torch.float32).squeeze()

Estim100%|██████████████████████████████| 100/100 [00:00<00:00, 3051.05it/s]


In [29]:
rbf_sigma_global = 0.25
k = 3

In [30]:
### FIXED FILTER
max_offset_sig = torch.tensor([k - 1]).to(device) # allow instantanous 
# max_offset_sig = torch.tensor([-1]).to(device)
sig_filter = torch.ones(size = (k, )).to(device)

# CCM analogy
ccm_filter = torch.ones(size = (k,))

N_length = x_norm.shape[0]
N = N_length - k + 1
l_train_masks_L100 = generate_mask_tensor(N, 100)

noise_scalar = torch.tensor([0.05], device = device)

# Null line D -> E

In [31]:
y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = e_norm.to(device), # Testing D -> E
                           x = d_norm.to(device),
                           max_pos_offset = max_offset_sig, 
                           device = device)

N = y_embeddings.shape[0]
E = y_embeddings.shape[1]

l_train_masks = generate_mask_tensor(N, 100)

x_gt_iaaft = torch.tensor(surrogates(x = x_gt.cpu(), ns = N, tol_pc = 10), dtype = torch.float32)

rho_l_indep = torch.empty(size = (1, 0)).to(device)

for l in range(N):   
    rho_L100, nlml_L100 = GP_ccm_sig(
            y_embeddings_train = y_embeddings[l_train_masks_L100[l]].unsqueeze(-1).to(device),
            y_embeddings_test = y_embeddings[ ~ l_train_masks_L100[l]].unsqueeze(-1).to(device),
            x_train = x_gt_iaaft[l, l_train_masks_L100[l]].to(device),
            x_test = x_gt_iaaft[l,  ~ l_train_masks_L100[l]].to(device),
            noise = noise_scalar,
            rbf_sigma = rbf_sigma_global,
            device = device)
            
    rho_l_indep = torch.concat((rho_l_indep, rho_L100.unsqueeze(0).unsqueeze(0)), dim = 1)

print(rho_l_indep.mean().item())
print(rho_l_indep.std().item())

quants = torch.tensor([0.05, 0.95]).to(device)
print("rho null upper (p95)", torch.quantile(rho_l_indep, quants)[1].item())

d_to_e_gpccm_null = torch.quantile(rho_l_indep, quants)[1].item()

Estim100%|██████████████████████████████| 398/398 [00:00<00:00, 3284.91it/s]


0.0035167515743523836
0.06731125712394714
rho null upper (p95) 0.12153831869363785


# Null model E -> D

In [32]:
y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = d_norm.to(device), # Testing E -> D (false)
                           x = e_norm.to(device),
                           max_pos_offset = max_offset_sig, 
                           device = device)

N = y_embeddings.shape[0]
E = y_embeddings.shape[1]

l_train_masks = generate_mask_tensor(N, 100)

x_gt_iaaft = torch.tensor(surrogates(x = x_gt.cpu(), ns = N, tol_pc = 10), dtype = torch.float32)

rho_l_indep = torch.empty(size = (1, 0)).to(device)

for l in range(N):   
    rho_L100, nlml_L100 = GP_ccm_sig(
            y_embeddings_train = y_embeddings[l_train_masks_L100[l]].unsqueeze(-1).to(device),
            y_embeddings_test = y_embeddings[ ~ l_train_masks_L100[l]].unsqueeze(-1).to(device),
            x_train = x_gt_iaaft[l, l_train_masks_L100[l]].to(device),
            x_test = x_gt_iaaft[l,  ~ l_train_masks_L100[l]].to(device),
            noise = noise_scalar,
            rbf_sigma = rbf_sigma_global,
            device = device)
            
    rho_l_indep = torch.concat((rho_l_indep, rho_L100.unsqueeze(0).unsqueeze(0)), dim = 1)

print(rho_l_indep.mean().item())
print(rho_l_indep.std().item())

quants = torch.tensor([0.05, 0.95]).to(device)
print("rho null upper (p95)", torch.quantile(rho_l_indep, quants)[1].item())

e_to_d_gpccm_null = torch.quantile(rho_l_indep, quants)[1].item()

Estim100%|██████████████████████████████| 398/398 [00:00<00:00, 3257.47it/s]


0.01339196227490902
0.06059470772743225
rho null upper (p95) 0.10550718009471893


# D -> E (true)

In [33]:
shifts = torch.arange(-8, 8 + 1, 1)
print(shifts)

# noise_scalar = torch.tensor([0.05], device = device)

rho_s_d_to_e = torch.zeros(size = (shifts.shape[0], 2))

for i, s in enumerate(shifts):
    # print(s.item())
    # print(i)

    y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = e_norm.to(device), # Testing X -> Y
                           x = d_norm.to(device),
                           max_pos_offset = s, 
                           device = device)
    
    # N now changes slightly
    N = y_embeddings.shape[0]
    E = y_embeddings.shape[1]

    l_train_masks = generate_mask_tensor(N, 100)

    rho_l = torch.empty(size = (1, 0)).to(device)

    for l in range(N):   
        rho, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                x_train = x_gt[l_train_masks[l]].to(device),
                x_test = x_gt[ ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = rbf_sigma_global, # 0.2 is good
                device = device)
                
        rho_l = torch.concat((rho_l, rho.unsqueeze(0).unsqueeze(0)), dim = 1)
     
    rho_s_d_to_e[i, 0] = rho_l.mean()
    rho_s_d_to_e[i, 1] = rho_l.std()

tensor([-8, -7, -6, -5, -4, -3, -2, -1,  0,  1,  2,  3,  4,  5,  6,  7,  8])


In [34]:
fig = go.Figure()
fig.update_layout(width = 600, height = 350)

fig.add_vrect(x0 = -(k - 1), x1 = 0.0, line_width = 0, fillcolor = "grey", opacity = 0.2)

fig.add_vrect(x0 = -8, x1 = -(k - 1), line_width = 0, fillcolor = "green", opacity = 0.1)
fig.add_vrect(x0 = 0., x1 = 8., line_width = 0, fillcolor = "red", opacity = 0.1)

fig.add_vline(x = -(k - 1), line_width = 1, fillcolor = "black", opacity = 0.6)
fig.add_vline(x = 0.0, line_width = 1, fillcolor = "black", opacity = 0.6)

fig.add_hline(y = d_to_e_gpccm_null, line_width = 1, line_dash = "dot", fillcolor = "black", opacity = 0.6)

fig.add_trace(go.Scatter(x = -shifts, y = rho_s_d_to_e[:, 0], # reversing the meaning of x
                    mode = 'lines+markers',
                    name = 'mean',
                    line_color = "#C00000"))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s_d_to_e[:, 0] + rho_s_d_to_e[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines'
))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s_d_to_e[:, 0] - rho_s_d_to_e[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines',
    fillcolor = 'rgba(243, 176, 210, 0.3)',
    fill = 'tonexty'
))

fig.update_layout(title = 'GP-CCM for D -> E',
                   xaxis_title = 'shift',
                   yaxis_title = 'rho')

# fig.update_layout(template = "plotly_white")
fig.update_layout(template = "simple_white")
fig.update_layout(font_family = "Lato")

fig.update_layout(xaxis_range = [-8.0, 8.0])
fig.update_layout(yaxis_range = [-0.1, 1.01])

fig.update_layout(legend = dict(x = 0, y = 1.01, bgcolor = "rgba(0,0,0,0)"))

fig.show()

# E -> D (false)

In [35]:
shifts = torch.arange(- 8, 8 + 1, 1)
print(shifts)

# noise_scalar = torch.tensor([0.05], device = device)

rho_s_e_to_d = torch.zeros(size = (shifts.shape[0], 2))

for i, s in enumerate(shifts):
    y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = x_norm.to(device), # Testing Y -> X
                           x = y_norm.to(device),
                           max_pos_offset = s, 
                           device = device)
    
    # N now changes slightly
    N = y_embeddings.shape[0]
    E = y_embeddings.shape[1]

    l_train_masks = generate_mask_tensor(N, 100)

    rho_l = torch.empty(size = (1, 0)).to(device)

    for l in range(N):   
        rho, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                x_train = x_gt[l_train_masks[l]].to(device),
                x_test = x_gt[ ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = rbf_sigma_global, # 0.2 is good
                device = device)
                
        rho_l = torch.concat((rho_l, rho.unsqueeze(0).unsqueeze(0)), dim = 1)
     
    rho_s_e_to_d[i, 0] = rho_l.mean()
    rho_s_e_to_d[i, 1] = rho_l.std()

tensor([-8, -7, -6, -5, -4, -3, -2, -1,  0,  1,  2,  3,  4,  5,  6,  7,  8])


In [36]:
gpccm_solid_color = 'rgba(36, 113, 163, 0.8)'
gpccm_fill_color = "rgba(36, 113, 163, 0.3)"

eccm_darker = "rgba(46, 134, 193, 1.0)"
eccm_solid_color = "rgba(30, 144, 255, 0.8)"
eccm_fill_color = "rgba(30, 144, 255, 0.3)"

In [37]:

fig = go.Figure()
fig.update_layout(width = 600, height = 350)

fig.add_vrect(x0 = -(k - 1), x1 = 0.0, line_width = 0, fillcolor = "grey", opacity = 0.2)

fig.add_vrect(x0 = -8, x1 = -(k - 1), line_width = 0, fillcolor = "green", opacity = 0.1)
fig.add_vrect(x0 = 0., x1 = 8., line_width = 0, fillcolor = "red", opacity = 0.1)

fig.add_vline(x = -(k - 1), line_width = 1, fillcolor = "black", opacity = 0.6)
fig.add_vline(x = 0.0, line_width = 1, fillcolor = "black", opacity = 0.6)

fig.add_hline(y = e_to_d_gpccm_null, line_dash = "dot", line_width = 1, fillcolor = "black", opacity = 0.6)

fig.add_trace(go.Scatter(x = -shifts, y = rho_s_e_to_d[:, 0], # reversing the meaning of x
                    mode = 'lines+markers',
                    name = 'mean rho +/- 1 sd',
                    line_color = "#C00000"))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s_e_to_d[:, 0] + rho_s_e_to_d[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines'
))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s_e_to_d[:, 0] - rho_s_e_to_d[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines',
    fillcolor = 'rgba(243, 176, 210, 0.3)',
    fill = 'tonexty'
))

fig.update_layout(title = 'GP-CCM for E -> D',
                   xaxis_title = 'shift',
                   yaxis_title = 'rho')

# fig.update_layout(template = "plotly_white")
fig.update_layout(template = "simple_white")
fig.update_layout(font_family = "Lato")

fig.update_layout(xaxis_range = [-8.0, 8.0])
fig.update_layout(yaxis_range = [-0.1, 1.01])

fig.update_layout(legend = dict(x = 0, y = 1.01, bgcolor = "rgba(0,0,0,0)"))

fig.show()

# ECCM baseline
# Nullline E -> D

In [38]:
# N different aurrogates
x_gt_iaaft = torch.tensor(surrogates(x = e_norm.cpu(), ns = e_norm.shape[0], tol_pc = 10), dtype = torch.float32)

l_train_masks = generate_mask_tensor(e_norm.shape[0], 100)

rho_l_ind = torch.empty(size = (1, 0)).to(device)
for l in range(x_gt_iaaft.shape[0]):
    rho = SP_CCM_iaaft(# y = d_norm.to(device), 
                       y = (d_norm + torch.randn(d_norm.shape[0]) * 0.05).to(device), 
                       x_gt_iaaft_selected = x_gt_iaaft[l].to(device), 
                       selected_train_mask = l_train_masks[l].to(device), 
                       max_offset = max_offset_sig.to(device),
                       ccmfilter = ccm_filter.to(device), 
                       device = device)
    
    rho_l_ind = torch.cat((rho_l_ind, rho.unsqueeze(0).unsqueeze(0)), dim = 1)

print("rho mean:", rho_l.mean().item())
print("rho ind mean:", rho_l_ind.mean().item())

# ranksums(rho_l.cpu().squeeze(), rho_l_null.cpu().squeeze())

quants = torch.tensor([0.05, 0.95]).to(device)
print("rho ind upper (p95)", torch.quantile(rho_l_ind, quants)[1].item())
e_to_d_eccm_null = torch.quantile(rho_l_ind, quants)[1].item()

Estim100%|██████████████████████████████| 400/400 [00:00<00:00, 4106.02it/s]


rho mean: -0.05220009386539459
rho ind mean: 0.013204053044319153
rho ind upper (p95) 0.11779437959194183


# Nullline D -> E

In [39]:
# N different aurrogates
x_gt_iaaft = torch.tensor(surrogates(x = d_norm.cpu(), ns = e_norm.shape[0], tol_pc = 10), dtype = torch.float32)

l_train_masks = generate_mask_tensor(d_norm.shape[0], 100)

rho_l_ind = torch.empty(size = (1, 0)).to(device)
for l in range(x_gt_iaaft.shape[0]):
    rho = SP_CCM_iaaft(y = e_norm.to(device), 
                       # y = (a_norm + torch.randn(c_norm.shape[0]) * 0.01).to(device), 
                       x_gt_iaaft_selected = x_gt_iaaft[l].to(device), 
                       selected_train_mask = l_train_masks[l].to(device), 
                       max_offset = max_offset_sig.to(device),
                       ccmfilter = ccm_filter.to(device), 
                       device = device)
    
    rho_l_ind = torch.cat((rho_l_ind, rho.unsqueeze(0).unsqueeze(0)), dim = 1)

print("rho mean:", rho_l.mean().item())
print("rho ind mean:", rho_l_ind.mean().item())

# ranksums(rho_l.cpu().squeeze(), rho_l_null.cpu().squeeze())

quants = torch.tensor([0.05, 0.95]).to(device)
print("rho ind upper (p95)", torch.quantile(rho_l_ind, quants)[1].item())
d_to_e_eccm_null = torch.quantile(rho_l_ind, quants)[1].item()

Estim100%|██████████████████████████████| 400/400 [00:00<00:00, 3969.77it/s]


rho mean: -0.05220009386539459
rho ind mean: 0.007678350433707237
rho ind upper (p95) 0.1278589367866516


# D -> E

In [40]:
shifts = torch.arange(-8, 8 + 1, 1)
print(shifts)

rho_s_d_to_e_eccm = torch.zeros(size = (shifts.shape[0], 2))

for i, s in enumerate(shifts):
    # One fast pass
    rho_l = run_SP_CCM(y = e_norm.to(device),
                        x = d_norm.to(device),
                        filter = ccm_filter.to(device),
                        max_offset = torch.tensor(s).to(device),
                        L = 100,
                        device = device)
    
    rho_s_d_to_e_eccm[i, 0] = rho_l.mean()
    rho_s_d_to_e_eccm[i, 1] = rho_l.std()

tensor([-8, -7, -6, -5, -4, -3, -2, -1,  0,  1,  2,  3,  4,  5,  6,  7,  8])


/tmp/ipykernel_1237667/1602912633.py:11: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).



In [41]:
fig = go.Figure()
fig.update_layout(width = 600, height = 350)

fig.add_vrect(x0 = -(k - 1), x1 = 0.0, line_width = 0, fillcolor = "grey", opacity = 0.2)

fig.add_vrect(x0 = -8, x1 = -(k - 1), line_width = 0, fillcolor = "green", opacity = 0.1)
fig.add_vrect(x0 = 0., x1 = 8., line_width = 0, fillcolor = "red", opacity = 0.1)

fig.add_vline(x = -(k - 1), line_width = 1, fillcolor = "black", opacity = 0.6)
fig.add_vline(x = 0.0, line_width = 1, fillcolor = "black", opacity = 0.6)

fig.add_hline(y = e_to_d_eccm_null, line_dash = "dot", line_width = 1, fillcolor = "black", opacity = 0.6)

fig.add_trace(go.Scatter(x = -shifts, y = rho_s_d_to_e_eccm[:, 0], # reversing the meaning of x
                    mode = 'lines+markers',
                    name = 'mean rho +/- 1 sd',
                    line_color = eccm_darker))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s_d_to_e_eccm[:, 0] + rho_s_d_to_e_eccm[:, 1].mul(1),
    marker = dict(color = eccm_solid_color),
    showlegend = False,
    mode = 'lines'
))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s_d_to_e_eccm[:, 0] - rho_s_d_to_e_eccm[:, 1].mul(1),
    marker = dict(color = eccm_solid_color),
    showlegend = False,
    mode = 'lines',
    fillcolor = eccm_fill_color,
    fill = 'tonexty'
))

fig.update_layout(title = 'GP-CCM for D -> E',
                   xaxis_title = 'shift',
                   yaxis_title = 'rho')

# fig.update_layout(template = "plotly_white")
fig.update_layout(template = "simple_white")
fig.update_layout(font_family = "Lato")

fig.update_layout(xaxis_range = [-8.0, 8.0])
fig.update_layout(yaxis_range = [-0.1, 1.01])

fig.update_layout(legend = dict(x = 0, y = 1.01, bgcolor = "rgba(0,0,0,0)"))

fig.show()

# E -> D (false)

In [42]:
shifts = torch.arange(-8, 8 + 1, 1)
print(shifts)

rho_s_e_to_d_eccm = torch.zeros(size = (shifts.shape[0], 2))

for i, s in enumerate(shifts):
    # One fast pass
    rho_l = run_SP_CCM(y = (d_norm + torch.randn(d_norm.shape[0]) * 0.01).to(device),
                        x = e_norm.to(device),
                        filter = ccm_filter.to(device),
                        max_offset = torch.tensor(s).to(device),
                        L = 100,
                        device = device)
    
    rho_s_e_to_d_eccm[i, 0] = rho_l.mean()
    rho_s_e_to_d_eccm[i, 1] = rho_l.std()

tensor([-8, -7, -6, -5, -4, -3, -2, -1,  0,  1,  2,  3,  4,  5,  6,  7,  8])


/tmp/ipykernel_1237667/1656086502.py:11: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).



In [43]:
fig = go.Figure()
fig.update_layout(width = 600, height = 350)

fig.add_vrect(x0 = -(k - 1), x1 = 0.0, line_width = 0, fillcolor = "grey", opacity = 0.2)

fig.add_vrect(x0 = -8, x1 = -(k - 1), line_width = 0, fillcolor = "green", opacity = 0.1)
fig.add_vrect(x0 = 0., x1 = 8., line_width = 0, fillcolor = "red", opacity = 0.1)

fig.add_vline(x = -(k - 1), line_width = 1, fillcolor = "black", opacity = 0.6)
fig.add_vline(x = 0.0, line_width = 1, fillcolor = "black", opacity = 0.6)

fig.add_hline(y = e_to_d_eccm_null, line_dash = "dot", line_width = 1, fillcolor = "black", opacity = 0.6)

fig.add_trace(go.Scatter(x = -shifts, y = rho_s_e_to_d_eccm[:, 0], # reversing the meaning of x
                    mode = 'lines+markers',
                    name = 'mean rho +/- 1 sd',
                    line_color = eccm_darker))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s_e_to_d_eccm[:, 0] + rho_s_e_to_d_eccm[:, 1].mul(1),
    marker = dict(color = eccm_solid_color),
    showlegend = False,
    mode = 'lines'
))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s_e_to_d_eccm[:, 0] - rho_s_e_to_d_eccm[:, 1].mul(1),
    marker = dict(color = eccm_solid_color),
    showlegend = False,
    mode = 'lines',
    fillcolor = eccm_fill_color,
    fill = 'tonexty'
))

fig.update_layout(title = 'GP-CCM for E -> D',
                   xaxis_title = 'shift',
                   yaxis_title = 'rho')

# fig.update_layout(template = "plotly_white")
fig.update_layout(template = "simple_white")
fig.update_layout(font_family = "Lato")

fig.update_layout(xaxis_range = [-8.0, 8.0])
fig.update_layout(yaxis_range = [-0.1, 1.01])

fig.update_layout(legend = dict(x = 0, y = 1.01, bgcolor = "rgba(0,0,0,0)"))

fig.show()